# E01 · Quanti minuti di sala dobbiamo riservare?

**Soluzione · Lezione 3**

La segreteria vuole il totale dei minuti da riservare per ogni sala. Se una sala compare più volte, sommiamo le sue prenotazioni valide. Ogni durata illeggibile, nulla o negativa conta come uno scarto.

**Dove lavorare.** Puoi usare direttamente la cartella del corso oppure una sottocartella a tua scelta. Salva insieme `richieste_sale.csv`, `prenotazioni.py` e `report_sale.py`. Apri il terminale in quella cartella: il programma cercherà lì il CSV e vi creerà `minuti_sale.csv`.

## Dati di partenza · richieste_sale.csv

```csv
sala,ore
Blu,2
Verde,1
Blu,1
Rossa,?
Verde,0
```

**Perché nella soluzione compare `encoding="utf-8"`?** Indicare la codifica è una buona pratica: rende esplicito come leggere e scrivere il testo su computer diversi. In questo esercizio era facoltativo: intestazioni e dati, compreso il carattere `?`, usano solo caratteri ASCII, compatibili con le normali codifiche predefinite dei computer del corso. Se in futuro aggiungi un nome come «Sala Città», salvare il CSV in UTF-8 e leggerlo con la stessa codifica permette di conservare anche la lettera accentata.

## 1 · Il modulo prenotazioni.py

Ogni oggetto conserva la sala e le ore di una prenotazione. Il metodo `minuti()` usa le ore di quell’oggetto.

In [ ]:
# in prenotazioni.py
class Prenotazione:                           # Definisce la classe che rappresenta una prenotazione.
    def __init__(self, sala, ore):            # Riceve la nuova istanza (self), il nome della sala e il numero di ore.
        self.sala = sala                      # Memorizza il nome della sala su questa prenotazione.
        self.ore = ore                        # Memorizza la durata in ore sulla stessa prenotazione.

    def minuti(self):                         # Definisce il metodo che calcola la durata di questa prenotazione.
        return self.ore * 60                  # Converte le ore in minuti e restituisce il risultato a chi ha chiamato il metodo.

## 2 · Il programma report_sale.py

Il programma legge le richieste con `csv.reader`. Per ogni riga valida crea una prenotazione e somma i minuti al totale della sala. Alla fine scrive i totali con `csv.writer`.

In [ ]:
# in report_sale.py
import csv                                    # Importa il modulo standard per leggere e scrivere righe CSV.
from pathlib import Path                      # Importa Path per indicare i percorsi dei due file.
import prenotazioni                           # Carica il nostro modulo prenotazioni.py dalla stessa cartella dello script.

prova = prenotazioni.Prenotazione("Prova", 2)  # Nel modulo prenotazioni usa la classe Prenotazione per creare un esempio di due ore.
print("Minuti di prova:", prova.minuti())     # Chiama il metodo sull’esempio e mostra 120 minuti.

origine = Path("richieste_sale.csv")          # Indica il CSV da leggere nella cartella di lavoro.
destinazione = Path("minuti_sale.csv")        # Indica il nome del report che il programma creerà nella stessa cartella.
totali = {}                                   # Prepara un dizionario: ogni chiave sarà una sala, ogni valore il suo totale in minuti.
scarti = 0                                    # Parte da zero prenotazioni scartate.

with origine.open(encoding="utf-8", newline="") as ingresso:  # Apre il CSV in lettura. with lo chiude a fine blocco e newline="" lascia gestire i fine riga a csv.
    lettore = csv.reader(ingresso)            # Prepara il lettore: ogni riga sarà una lista di stringhe.
    next(lettore, None)      # Consuma la prima riga, sala,ore. Se il file è vuoto, next restituisce None.
    for riga in lettore:        # Visita una prenotazione per volta a partire dalla prima riga di dati.
        if len(riga) < 2:        # Controlla se manca la seconda colonna, quella delle ore.
            scarti += 1        # Conta la riga incompleta come uno scarto.
            continue           # Passa alla riga successiva prima di provare a leggere le ore.
        try:                # Prova a convertire il testo della durata in un numero intero.
            ore = int(riga[1])        # Legge la seconda colonna: per esempio, la stringa "2" diventa il numero 2.
        except ValueError:         # Gestisce una durata illeggibile, come il punto interrogativo di Rossa.
            scarti += 1           # Aggiunge uno scarto per la durata illeggibile.
            continue            # Passa alla riga seguente senza creare una prenotazione.
        if ore <= 0:            # Controlla se le ore sono zero o negative.
            scarti += 1            # Conta anche questa durata come uno scarto.
            continue             # Passa alla riga seguente senza sommare minuti.
        voce = prenotazioni.Prenotazione(riga[0], ore)  # Crea una prenotazione valida con la sala della prima colonna e le ore convertite.
        if voce.sala in totali:               # Verifica se è già presente un totale per questa sala.
            totali[voce.sala] += voce.minuti()  # Aggiunge al totale precedente i minuti della nuova prenotazione.
        else:                                 # Gestisce la prima prenotazione valida di questa sala.
            totali[voce.sala] = voce.minuti()  # Registra i primi minuti della sala nel dizionario.

with destinazione.open("w", encoding="utf-8", newline="") as uscita:  # Apre il report in scrittura. La modalità "w" lo ricrea a ogni esecuzione.
    scrittore = csv.writer(uscita)            # Prepara lo scrittore CSV collegato al file di uscita.
    scrittore.writerow(["sala", "minuti"])    # Scrive l’intestazione con i nomi delle due colonne.
    for sala, minuti in totali.items():       # Prende dal dizionario una sala e il suo totale per volta.
        scrittore.writerow([sala, minuti])    # Scrive una riga del report con la sala e i suoi minuti.

with destinazione.open(encoding="utf-8", newline="") as uscita:  # Riapre il report appena scritto per controllare il contenuto salvato.
    for riga in csv.reader(uscita):           # Legge tutte le righe del report, compresa l’intestazione.
        print(riga)                           # Mostra a schermo la riga appena riletta.
print("Scarti:", scarti)                      # Mostra il numero complessivo di prenotazioni scartate.

## 3 · Esecuzione e verifica

Dopo aver salvato i file, esegui dal terminale della cartella scelta:

```powershell
python report_sale.py
```

Nel file `minuti_sale.csv` trovi:

```csv
sala,minuti
Blu,180
Verde,60
```

**Blu** compare due volte in ingresso: 2 ore + 1 ora = **180 minuti**. Per **Verde** è valida soltanto la prenotazione di 1 ora, quindi il totale è **60 minuti**. Il terminale mostra `Scarti: 2`: il valore `?` di Rossa e lo zero di Verde.

**Seconda prova.** Aggiungi `Verde,1` in fondo a `richieste_sale.csv` e rilancia lo stesso comando. Blu resta a 180 minuti, Verde passa a 120 e gli scarti restano 2. Il report viene riscritto a partire dai dati di ingresso.